In [1]:
pwd()

"C:\\Users\\Emmanuel Ganyo"

In [2]:
cd("C:/DataSc")

In [3]:
pwd()

"C:\\DataSc"

In [4]:
using Pkg

In [5]:
using CSV, DataFrames, GLM, CategoricalArrays, StatsBase, Random, Plots

In [6]:
df = CSV.read("Churn_Modelling.csv", DataFrame)

Row,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
,Int64,Int64,DataStri…,Int64,DataStri…,DataStri…,Int64,Int64,Float64,Int64,Int64,Int64,Float64,Int64
1,1,15634602,Hargrave,619,France,Female,42,2,0.0,1,1,1,1.01349e5,1
2,2,15647311,Hill,608,Spain,Female,41,1,83807.9,1,0,1,1.12543e5,0
3,3,15619304,Onio,502,France,Female,42,8,1.59661e5,3,1,0,1.13932e5,1
4,4,15701354,Boni,699,France,Female,39,1,0.0,2,0,0,93826.6,0
5,5,15737888,Mitchell,850,Spain,Female,43,2,1.25511e5,1,1,1,79084.1,0
6,6,15574012,Chu,645,Spain,Male,44,8,1.13756e5,2,1,0,1.49757e5,1
7,7,15592531,Bartlett,822,France,Male,50,7,0.0,2,1,1,10062.8,0
8,8,15656148,Obinna,376,Germany,Female,29,4,1.15047e5,4,1,0,1.19347e5,1
9,9,15792365,He,501,France,Male,44,4,1.42051e5,2,0,1,74940.5,0


In [7]:
first(df, 5)
last(df, 5)

Row,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
,Int64,Int64,DataStri…,Int64,DataStri…,DataStri…,Int64,Int64,Float64,Int64,Int64,Int64,Float64,Int64
1,9996,15606229,Obijiaku,771,France,Male,39,5,0.0,2,1,0,96270.6,0
2,9997,15569892,Johnstone,516,France,Male,35,10,57369.6,1,1,1,1.017e5,0
3,9998,15584532,Liu,709,France,Female,36,7,0.0,1,0,1,42085.6,1
4,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.3,2,1,0,92888.5,1
5,10000,15628319,Walker,792,France,Female,28,4,1.30143e5,1,1,0,38190.8,0


In [8]:
df = select(df, Not([:RowNumber, :CustomerId, :Surname]))

Row,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
,Int64,DataStri…,DataStri…,Int64,Int64,Float64,Int64,Int64,Int64,Float64,Int64
1,619,France,Female,42,2,0.0,1,1,1,1.01349e5,1
2,608,Spain,Female,41,1,83807.9,1,0,1,1.12543e5,0
3,502,France,Female,42,8,1.59661e5,3,1,0,1.13932e5,1
4,699,France,Female,39,1,0.0,2,0,0,93826.6,0
5,850,Spain,Female,43,2,1.25511e5,1,1,1,79084.1,0
6,645,Spain,Male,44,8,1.13756e5,2,1,0,1.49757e5,1
7,822,France,Male,50,7,0.0,2,1,1,10062.8,0
8,376,Germany,Female,29,4,1.15047e5,4,1,0,1.19347e5,1
9,501,France,Male,44,4,1.42051e5,2,0,1,74940.5,0


In [9]:
df.Exited = Int.(df.Exited);

In [10]:
df.Geography = categorical(df.Geography)

10000-element CategoricalArray{DataStrings.DataString,1,UInt32}:
 "France"
 "Spain"
 "France"
 "France"
 "Spain"
 "Spain"
 "France"
 "Germany"
 "France"
 "France"
 "France"
 "Spain"
 "France"
 ⋮
 "France"
 "Spain"
 "Germany"
 "France"
 "Spain"
 "France"
 "France"
 "France"
 "France"
 "France"
 "Germany"
 "France"

In [11]:
df.Gender    = categorical(df.Gender)

10000-element CategoricalArray{DataStrings.DataString,1,UInt32}:
 "Female"
 "Female"
 "Female"
 "Female"
 "Female"
 "Male"
 "Male"
 "Female"
 "Male"
 "Male"
 "Male"
 "Male"
 "Female"
 ⋮
 "Male"
 "Male"
 "Male"
 "Female"
 "Male"
 "Male"
 "Female"
 "Male"
 "Male"
 "Female"
 "Male"
 "Female"

In [12]:
 Random.seed!(42)

TaskLocalRNG()

In [13]:
function train_test_split(df, p=0.8)
           n = nrow(df)
           idx = shuffle(1:n)
           n_train = floor(Int, p * n)
           return df[idx[1:n_train], :], df[idx[n_train+1:end], :]
   end

train_test_split (generic function with 2 methods)

In [14]:
train, test = train_test_split(df);

In [15]:
first(train, 10)
last(train, 10)

Row,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
,Int64,Cat…,Cat…,Int64,Int64,Float64,Int64,Int64,Int64,Float64,Int64
1,452,France,Female,35,7,0.0,2,1,0,1.64242e5,0
2,731,Spain,Male,46,10,0.0,2,1,0,1.53015e5,0
3,616,Spain,Male,31,7,76665.7,2,1,1,1.63809e5,0
4,510,France,Male,63,8,0.0,2,1,1,1.15292e5,0
5,467,Spain,Male,39,5,0.0,2,1,1,7415.96,0
6,556,Spain,Female,48,8,1.68522e5,1,1,1,1.5131e5,0
7,586,France,Male,29,3,0.0,2,1,1,1.42239e5,0
8,696,Spain,Male,47,1,1.06759e5,1,1,1,80591.2,0
9,651,Germany,Male,26,5,1.47037e5,1,0,0,1.41763e5,0


In [16]:
formula = @formula(Exited ~ CreditScore + Geography + Gender + Age + Tenure + Balance + NumOfProducts + HasCrCard +IsActiveMember + EstimatedSalary)

FormulaTerm
Response:
  Exited(unknown)
Predictors:
  CreditScore(unknown)
  Geography(unknown)
  Gender(unknown)
  Age(unknown)
  Tenure(unknown)
  Balance(unknown)
  NumOfProducts(unknown)
  HasCrCard(unknown)
  IsActiveMember(unknown)
  EstimatedSalary(unknown)

In [17]:
model = glm(formula, train, Binomial(), LogitLink())

StatsModels.TableRegressionModel{GeneralizedLinearModel{GLM.GlmResp{Vector{Float64}, Binomial{Float64}, LogitLink}, GLM.DensePredChol{Float64, LinearAlgebra.CholeskyPivoted{Float64, Matrix{Float64}, Vector{Int64}}}}, Matrix{Float64}}

Exited ~ 1 + CreditScore + Geography + Gender + Age + Tenure + Balance + NumOfProducts + HasCrCard + IsActiveMember + EstimatedSalary

Coefficients:
──────────────────────────────────────────────────────────────────────────────────────────
                           Coef.   Std. Error       z  Pr(>|z|)    Lower 95%     Upper 95%
──────────────────────────────────────────────────────────────────────────────────────────
(Intercept)         -3.40739      0.272847     -12.49    <1e-35  -3.94216     -2.87262
CreditScore         -0.000721987  0.000311216   -2.32    0.0203  -0.00133196  -0.000112015
Geography: Germany   0.760235     0.075484      10.07    <1e-23   0.612289     0.908181
Geography: Spain     0.0125559    0.0787459      0.16    0.8733  -0.141783   